# PointMaze continual RL — our method vs. eight baselines

Ten tasks, one chain per method, resumable across Kaggle sessions.

**How to use this notebook**

1. Run cells 1–3 (setup + smoke tests). If the smoke tests fail, stop: nothing below will be valid.
2. Cell 4 restores checkpoints from a previous session, if you attached one.
3. **Cell 5 is the only cell you normally edit.** Put the methods you want *this* session to run in `RUN`.
   It prints the plan and a measured time estimate and trains nothing, so you can decide before
   spending a session.
4. Cell 6 trains. It stops cleanly before the session limit and leaves resumable state.
5. Cells 7–8 score, plot, and package the output for the next session.

**Splitting work with someone else.** Each name in `RUN` is an independent chain writing under its own
`save_root/suite/tag/method/seed_*` path. Give your collaborator a different `RUN` list; at the end,
attach both output datasets to one session and cell 7 will score everything it finds.

## 1. Setup

In [ ]:
import os
import sys
import pathlib
import shutil
import subprocess
import json
import time

WORK = pathlib.Path("/kaggle/working")
CODE = WORK / "pointmaze"
INPUT = pathlib.Path("/kaggle/input")

def find_source():
    """Locate the pointmaze package: working dir first, then any attached dataset
    (searched recursively, since Kaggle sometimes nests datasets one level deeper,
    e.g. /kaggle/input/datasets/<slug>/pointmaze)."""
    if (CODE / "run_continual_benchmark.py").is_file():
        return CODE
    for candidate in INPUT.rglob("run_continual_benchmark.py"):
        return candidate.parent
    return None

src = find_source()
if src is None:
    raise FileNotFoundError(
        "Could not find run_continual_benchmark.py inside /kaggle/input. "
        "Attach the pointmaze dataset/zip contents under Add Data first."
    )

if src != CODE:
    if CODE.exists():
        shutil.rmtree(CODE)
    shutil.copytree(src, CODE)
    print(f"Copied code from: {src}")

os.chdir(CODE)
sys.path.insert(0, str(CODE))

print("Source:", src)
print("Working directory:", CODE)
print("Files:", list(CODE.iterdir())[:10])


In [ ]:
# Dependencies. Kaggle images already carry torch, numpy and gymnasium; this only
# reports what is present rather than reinstalling and risking a version conflict.
import importlib

for name in ("numpy", "torch", "gymnasium", "matplotlib"):
    try:
        mod = importlib.import_module(name)
        print(f"{name:12s} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{name:12s} MISSING")

import torch
print("cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")

## 2. Smoke tests — run these before spending a session

In [ ]:
# Environment only: a few seconds, no torch. Catches every env-level problem.
print(subprocess.run([sys.executable, "pointmaze_smoke.py"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Static checks: imports resolve, and every CLI flag the runner passes exists.
for script in ("verify_static.py", "verify_flags.py"):
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(f"--- {script} (exit {r.returncode}) ---")
    print(r.stdout or r.stderr)

In [ ]:
# Tiny end-to-end run of EVERY method (~10 minutes on CPU). Strongly recommended
# once per environment: it exercises construction, a task boundary, the merge,
# saving, reloading and scoring for all ten configurations.
RUN_FULL_SMOKE = True   # set False to skip once you trust the image

if RUN_FULL_SMOKE:
    r = subprocess.run([sys.executable, "baseline_smoke.py", "--method", "all"],
                       capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise SystemExit("smoke test failed - do not start a real run")

## 3. Restore a previous session

Attach the previous run's notebook output under **Add data → Your work → Notebook output**.
Everything already finished is skipped by manifest, so a chain that died at task 7 resumes at task 7.

In [ ]:
# Where results live. These are the only paths that need to survive a session.
SAVE_ROOT     = WORK / "agents_pointmaze"     # checkpoints (the expensive part)
RUNS_ROOT     = WORK / "runs_pointmaze"       # scalar logs
RESULTS_ROOT  = WORK / "results_pointmaze"    # chain + metric JSONs
ANALYSIS_ROOT = WORK / "analysis_pointmaze"   # task-boundary snapshots
PLOTS_ROOT    = WORK / "plots_pointmaze"

restored = 0
for base in sorted(pathlib.Path("/kaggle/input").glob("*")):
    for name, dest in (("agents_pointmaze", SAVE_ROOT),
                       ("results_pointmaze", RESULTS_ROOT),
                       ("runs_pointmaze", RUNS_ROOT),
                       ("analysis_pointmaze", ANALYSIS_ROOT)):
        source = base / name
        if source.is_dir():
            shutil.copytree(source, dest, dirs_exist_ok=True)
            restored += 1
            print(f"restored {source} -> {dest}")

if restored == 0:
    print("no previous session found; starting fresh")

def completed_stages(save_root, suite, tag):
    """Count finished task checkpoints per (method, seed)."""
    done = {}
    root = pathlib.Path(save_root) / suite / tag
    if not root.is_dir():
        return done
    for method_dir in sorted(root.iterdir()):
        for seed_dir in sorted(method_dir.glob("seed_*")):
            n = sum(
                1 for d in seed_dir.glob("seq_*/task_*")
                if (d / "manifest.json").is_file()
                and ((d / "agent.pt").is_file() or (d / "policy_snapshot.pt").is_file())
            )
            if n:
                done[f"{method_dir.name}/{seed_dir.name}"] = n
    return done

## 4. Choose what to run — **edit this cell**

`RUN` is the list of method chains this session will train. Nothing is trained here; the cell
prints the plan, what is already finished, and a *measured* time estimate.

Available: `Ours`, `Ours-parameter`, `CKA-RL`, `FT-N`, `ProgNet`, `PackNet`, `MaskNet`,
`CReLUs`, `CompoNet`, `CbpNet`.

`Ours` is the reported configuration: condition 4 (combined) — weight-delta vectors, alpha-mass,
behavioural-KL distillation merge — composed in **policy** space.

In [ ]:
# ----------------------------- EDIT ME -----------------------------
RUN = ["Ours"]
# each collaborator sets exactly one entry from:
#   "Ours", "CKA-RL", "FT-N", "ProgNet", "PackNet", "MaskNet",
#   "CReLUs", "CompoNet", "CbpNet"   (and "Ours-parameter" if you want it too)

SUITE  = "pointmaze_goal"      # or "pointmaze_goal_dyn"
SEEDS  = [1]                   # add 2, 3 for publication-grade error bars
TAG    = "main"

# TOTAL_TIMESTEPS / DISTILL_EXTRA_STEPS are picked automatically below from a
# real, measured SAC-update speed on THIS machine -- do not hand-edit them.
# TEST_ADAPT_STEPS is fixed (see README, "Test-time adaptation budget").
TEST_ADAPT_STEPS    = 6_000    # 20 episodes x 300-step horizon; see README
SESSION_BUDGET_MIN  = 480      # stop launching new tasks after this many minutes;
                                # set this close to (Kaggle's real session cap - ~20 min)
# -------------------------------------------------------------------

import tasks
report = tasks.validate_suite(SUITE)
print(f"suite {SUITE}: {report['num_tasks']} tasks, order {report['order']}")
print(f"  families {''.join(report['families'])} (no two consecutive share a family)")
print(f"  within-family shared path prefix {report['within_family_shared_prefix']}")
print(f"  across-family                    {report['across_family_shared_prefix']}")

done = completed_stages(SAVE_ROOT, SUITE, TAG)
print("\nalready finished:", json.dumps(done, indent=2) if done else "nothing")

# --- measured time estimate -------------------------------------------------
import torch, numpy as np, time as _time
from pointmaze_env import OBS_DIM, ACT_DIM
from baselines.agents import FtNAgent
from replay_buffer import ReplayBuffer

def calibrate(n=150, batch_size=256):
    """Time real SAC updates on this machine instead of guessing."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = FtNAgent(obs_dim=OBS_DIM, act_dim=ACT_DIM, num_tasks=1).to(dev)
    agent.set_task(0)
    buf = ReplayBuffer(5000, OBS_DIM, ACT_DIM)
    rng = np.random.default_rng(0)
    for _ in range(1000):
        buf.add(rng.normal(size=OBS_DIM), rng.uniform(-1, 1, ACT_DIM), -1.0,
                rng.normal(size=OBS_DIM), 0.0)
    opt = torch.optim.Adam(agent.parameters(), lr=3e-4)
    for _ in range(10):                       # warm up kernels / autotune
        o, a, r, n2, d = buf.sample(batch_size, dev)
        f = agent.encode(o); q1, q2 = agent.critic(f, a)
        loss = (q1.mean() + q2.mean())
        opt.zero_grad(); loss.backward(); opt.step()
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = _time.time()
    for _ in range(n):
        o, a, r, n2, d = buf.sample(batch_size, dev)
        f = agent.encode(o)
        dist = agent.distribution_from_features(f)
        act, logp = dist.rsample_with_log_prob()
        q1, q2 = agent.critic(f, a)
        loss = q1.mean() + q2.mean() + logp.mean() + agent.critic.min_q(f, act).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    if dev.type == "cuda":
        torch.cuda.synchronize()
    return (_time.time() - t0) / n

per_update = calibrate()
n_tasks = report["num_tasks"]

# --- auto-pick the largest Delta that safely finishes under Kaggle's session limit ---
# TARGET_HOURS is deliberately well under 12: calibrate() only measures the
# lightweight FtN network, but Ours / PackNet / CompoNet / MaskNet / CbpNet all do
# extra per-step work (mixture/pool math, masking, growing attention over prior
# modules) that FtN does not -- so their real per-update cost is higher than what
# was just measured. SAFETY absorbs that gap plus eval/merge/scratch-reference
# overhead that is not in this training-loop-only estimate.
TARGET_HOURS = 9.0
SAFETY       = 1.4

budget_seconds = TARGET_HOURS * 3600
TOTAL_TIMESTEPS = int(budget_seconds / (n_tasks * per_update * 1.12 * SAFETY))
TOTAL_TIMESTEPS = (TOTAL_TIMESTEPS // 10_000) * 10_000        # round down to a clean number
DISTILL_EXTRA_STEPS = max(4_000, int(0.04 * TOTAL_TIMESTEPS)) # keep tail ~4% of the budget

seconds_per_chain = TOTAL_TIMESTEPS * n_tasks * per_update * 1.12
print(f"\nmeasured {per_update*1000:.2f} ms / update (FtN network -- a lower bound; "
      f"heavier methods will run slower than this estimate)")
print(f"chosen TOTAL_TIMESTEPS = {TOTAL_TIMESTEPS:,}  (DISTILL_EXTRA_STEPS = {DISTILL_EXTRA_STEPS:,})")
print(f"estimated {seconds_per_chain/3600:.2f} h per chain "
      f"(raw training loop; add eval/merge/scratch-reference on top)")
print(
    "\n>>> Run this cell ONCE (e.g. on your own session), then copy the two numbers\n"
    ">>> above into every collaborator's notebook as fixed literals -- do NOT let\n"
    ">>> each person re-run calibrate() and pick their own slightly different value;\n"
    ">>> TOTAL_TIMESTEPS/DISTILL_EXTRA_STEPS must be identical across everyone for\n"
    ">>> the checkpoints to merge into one comparison at the end."
)

todo = [(m, s) for m in RUN for s in SEEDS
        if done.get(f"{m}/seed_{s}", 0) < n_tasks]
remaining = sum(n_tasks - done.get(f"{m}/seed_{s}", 0) for m, s in todo)
print(f"\nthis session: {len(todo)} chain(s), {remaining} task(s) left to train")
print(f"estimated {remaining * seconds_per_chain / n_tasks / 60:.0f} min "
      f"(session budget {SESSION_BUDGET_MIN} min)")
if remaining * seconds_per_chain / n_tasks / 60 > SESSION_BUDGET_MIN:
    print("\n  -> will not finish in one session. That is fine: cell 6 stops cleanly\n"
          "     before the budget and cell 8 saves resumable state.")


## 5. Train

In [ ]:
# Trains the chains in RUN, skipping anything already finished.
# --max-seconds makes the runner stop launching NEW tasks near the session
# limit, so the session ends with a consistent, resumable checkpoint set
# instead of being killed mid-write.
cmd = [
    sys.executable, "run_continual_benchmark.py",
    "--methods", *RUN,
    "--suite", SUITE,
    "--seeds", *[str(s) for s in SEEDS],
    f"--tag={TAG}",
    f"--total-timesteps={TOTAL_TIMESTEPS}",
    f"--distill-extra-steps={DISTILL_EXTRA_STEPS}",
    f"--test-adapt-steps={TEST_ADAPT_STEPS}",
    f"--save-root={SAVE_ROOT}",
    f"--runs-root={RUNS_ROOT}",
    f"--results-root={RESULTS_ROOT}",
    f"--analysis-root={ANALYSIS_ROOT}",
    f"--max-seconds={SESSION_BUDGET_MIN * 60}",
    "--skip-eval",          # score separately in cell 7, so training time is all training
    "--cuda",
]
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nexit code: {proc.returncode}")

## 6. Score and plot

Scoring is separate from training so that a session which runs out of time still leaves usable
checkpoints. Only the diagonal and the final row of each performance matrix are evaluated — the
other 80 cells are never read by any reported metric.

In [ ]:
# From-scratch references for the forward-transfer denominator.
# Seed 201 deliberately does not overlap the continual seeds.
SCRATCH_JSON = RESULTS_ROOT / "scratch_reference.json"
RUN_SCRATCH = True     # ~1 chain's worth of compute; needed only once per suite

if RUN_SCRATCH and not SCRATCH_JSON.is_file():
    r = subprocess.run(
        [sys.executable, "scratch_baselines.py",
         f"--suite={SUITE}", "--seeds", "201",
         f"--total-timesteps={TOTAL_TIMESTEPS}",
         f"--save-root={SAVE_ROOT}", f"--runs-root={RUNS_ROOT}",
         f"--out={SCRATCH_JSON}"],
        capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
elif SCRATCH_JSON.is_file():
    print(f"using existing {SCRATCH_JSON}")

In [ ]:
# Score every chain that is complete. Re-runnable and idempotent.
cmd = [
    sys.executable, "run_continual_benchmark.py",
    "--methods", *RUN,
    "--suite", SUITE,
    "--seeds", *[str(s) for s in SEEDS],
    f"--tag={TAG}",
    f"--total-timesteps={TOTAL_TIMESTEPS}",
    f"--distill-extra-steps={DISTILL_EXTRA_STEPS}",
    f"--test-adapt-steps={TEST_ADAPT_STEPS}",
    f"--save-root={SAVE_ROOT}",
    f"--runs-root={RUNS_ROOT}",
    f"--results-root={RESULTS_ROOT}",
    f"--analysis-root={ANALYSIS_ROOT}",
    "--cuda",
]
if SCRATCH_JSON.is_file():
    cmd.append(f"--scratch-reference={SCRATCH_JSON}")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nexit code: {proc.returncode}")

In [ ]:
# Results table across every method present (including a collaborator's, if
# their output dataset was attached in cell 4).
import glob
import numpy as np

rows = []
for path in sorted(glob.glob(str(RESULTS_ROOT / f"{SUITE}__*__metrics.json"))):
    with open(path) as f:
        m = json.load(f)
    rows.append((
        m.get("method"), m.get("seed"),
        m.get("average_final"), m.get("average_peak"),
        m.get("forgetting"), m.get("forward_transfer"),
        m.get("average_final_success"),
    ))

if rows:
    header = f"{'method':<16}{'seed':>5}{'final':>9}{'peak':>9}{'forget':>9}{'FT':>8}{'succ':>7}"
    print(header)
    print("-" * len(header))
    for r in sorted(rows):
        vals = ["" if v is None or (isinstance(v, float) and not np.isfinite(v))
                else (f"{v:.3f}" if isinstance(v, float) else str(v)) for v in r[2:]]
        print(f"{r[0]:<16}{r[1]:>5}{vals[0]:>9}{vals[1]:>9}{vals[2]:>9}{vals[3]:>8}{vals[4]:>7}")
else:
    print("no metrics yet")

In [ ]:
r = subprocess.run([sys.executable, "plots.py",
                    f"--results-root={RESULTS_ROOT}", f"--suite={SUITE}",
                    f"--out={PLOTS_ROOT}"], capture_output=True, text=True)
print(r.stdout or r.stderr)

from IPython.display import Image, display
for png in sorted(PLOTS_ROOT.glob("*.png"))[:6]:
    print(png.name)
    display(Image(str(png)))

## 7. Save for the next session

Kaggle keeps `/kaggle/working` as the notebook output. Commit the notebook, then in the next
session attach this run under **Add data → Your work → Notebook output**, and cell 4 restores it.

The cell below prunes anything large that is not needed to resume, so the output stays under
Kaggle's size limit.

In [ ]:
# Analysis snapshots are diagnostics, not resume state. Drop them if the output
# is large; checkpoints and results are what a later session actually needs.
PRUNE_ANALYSIS = True
MAX_OUTPUT_GB = 18.0

def dir_size_gb(path):
    path = pathlib.Path(path)
    if not path.is_dir():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024 ** 3)

for name, path in (("checkpoints", SAVE_ROOT), ("results", RESULTS_ROOT),
                   ("logs", RUNS_ROOT), ("analysis", ANALYSIS_ROOT),
                   ("plots", PLOTS_ROOT)):
    print(f"{name:<12} {dir_size_gb(path):6.2f} GB")

total = sum(dir_size_gb(p) for p in (SAVE_ROOT, RESULTS_ROOT, RUNS_ROOT, ANALYSIS_ROOT, PLOTS_ROOT))
print(f"{'total':<12} {total:6.2f} GB")

if PRUNE_ANALYSIS and total > MAX_OUTPUT_GB and ANALYSIS_ROOT.is_dir():
    shutil.rmtree(ANALYSIS_ROOT)
    print(f"\npruned {ANALYSIS_ROOT} to fit the output limit "
          "(checkpoints and results are untouched; only diagnostics were dropped)")

# Also drop the code copy: it comes from the zip, not from this run.
if CODE.is_dir() and (WORK / "pointmaze").exists():
    print("\nnote: /kaggle/working/pointmaze holds the source copy; that is fine to keep.")

print("\nfinished chains:")
print(json.dumps(completed_stages(SAVE_ROOT, SUITE, TAG), indent=2))